# CanastaCL — 01 · Análisis Exploratorio (EDA)

**Objetivo:** primer recorrido por el dataset `precio_consumidor_2026.csv` para entender estructura, calidad y distribuciones antes de limpiar y modelar.

**Secciones:**
1. Setup e importaciones
2. Carga del dataset
3. Inspección estructural (shape, dtypes, nulls)
4. Calidad de datos
5. Estadística descriptiva
6. Distribución por dimensiones (región, grupo, tipo de comercio)
7. Series temporales preliminares
8. Hallazgos y próximos pasos

## 1. Setup e importaciones

In [ ]:
import sys
from pathlib import Path

# Permitir importar el paquete src/ desde el notebook
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (11, 5)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

from src.data_loader import load_raw

## 2. Carga del dataset

Usamos la función `load_raw` definida en `src/data_loader.py` para mantener la lógica de ingesta centralizada.

In [ ]:
df = load_raw()
print(f"Filas: {len(df):,} | Columnas: {df.shape[1]}")
df.head()

## 3. Inspección estructural

In [ ]:
df.info()

In [ ]:
# Tipos y muestra de valores únicos
resumen = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_unicos": df.nunique(),
    "n_nulos": df.isna().sum(),
    "%_nulos": (df.isna().mean() * 100).round(2),
})
resumen

## 4. Calidad de datos

Validamos rangos, fechas y consistencia entre `Precio minimo`, `Precio promedio` y `Precio maximo`.

In [ ]:
# Rango temporal
print("Fecha inicio:", df["Fecha inicio"].min(), "->", df["Fecha inicio"].max())
print("Fecha termino:", df["Fecha termino"].min(), "->", df["Fecha termino"].max())
print("Semanas distintas:", df["Semana"].nunique())

In [ ]:
# Coherencia de precios: min <= promedio <= max
incoherentes = df[
    (df["Precio minimo"] > df["Precio promedio"]) |
    (df["Precio promedio"] > df["Precio maximo"])
]
print(f"Filas con precios incoherentes: {len(incoherentes):,}")
incoherentes.head()

In [ ]:
# Filas duplicadas exactas
print("Duplicados exactos:", df.duplicated().sum())

## 5. Estadística descriptiva

In [ ]:
df[["Precio minimo", "Precio promedio", "Precio maximo"]].describe()

In [ ]:
# Distribución (escala log porque hay productos muy caros y muy baratos)
fig, ax = plt.subplots(1, 1)
sns.histplot(data=df, x="Precio promedio", bins=80, log_scale=(True, False), ax=ax)
ax.set_title("Distribución de Precio promedio (log)")
ax.set_xlabel("Precio promedio (CLP) — escala log")
plt.show()

## 6. Distribución por dimensiones

In [ ]:
# Conteo por región
conteo_region = df["Region"].value_counts()
conteo_region

In [ ]:
# Conteo por tipo de punto de monitoreo
df["Tipo de punto monitoreo"].value_counts()

In [ ]:
# Conteo por grupo de producto
df["Grupo"].value_counts().head(20)

In [ ]:
# Boxplot de precios por grupo (top 10 grupos más frecuentes)
top_grupos = df["Grupo"].value_counts().head(10).index
fig, ax = plt.subplots(figsize=(13, 6))
sns.boxplot(
    data=df[df["Grupo"].isin(top_grupos)],
    x="Grupo", y="Precio promedio", ax=ax,
)
ax.set_yscale("log")
ax.set_title("Precio promedio por grupo (top 10) — escala log")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 7. Series temporales preliminares

Vista agregada del precio promedio por semana, separando por región para detectar diferencias regionales.

In [ ]:
serie_region = (
    df.groupby(["Fecha inicio", "Region"], as_index=False)["Precio promedio"]
      .mean()
)

fig, ax = plt.subplots(figsize=(13, 6))
sns.lineplot(data=serie_region, x="Fecha inicio", y="Precio promedio", hue="Region", ax=ax)
ax.set_title("Precio promedio semanal por región (todos los productos)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Producto específico: Pan (un commodity de comparación clásica)
pan = df[df["Producto"].str.contains("Pan", case=False, na=False)]
print("Productos detectados con 'Pan':", pan["Producto"].unique())

if not pan.empty:
    serie_pan = (
        pan.groupby(["Fecha inicio", "Region"], as_index=False)["Precio promedio"]
           .mean()
    )
    fig, ax = plt.subplots(figsize=(13, 5))
    sns.lineplot(data=serie_pan, x="Fecha inicio", y="Precio promedio", hue="Region", ax=ax)
    ax.set_title("Precio promedio del Pan por región")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.show()

## 8. Hallazgos preliminares y próximos pasos

_Completar tras ejecutar el notebook:_

- [ ] Volumetría: filas, columnas, rango temporal cubierto.
- [ ] Calidad: filas con precios incoherentes, duplicados, nulos.
- [ ] Productos y grupos con mayor cobertura.
- [ ] Diferencias regionales evidentes en distribución.
- [ ] Estacionalidad o tendencia visible en series.

**Próximo notebook (`02_limpieza.ipynb`):** definir reglas de limpieza, normalizar `Unidad`, filtrar incoherencias y generar `data/processed/precios_clean.parquet`.